# Prepare System Collections

This notebook inspects the SCOPE Systems in the SCO ATLAS `Data` directory and prepares the CSV system collections used by the subsequent analysis notebooks.

## Configuration

Requirements:

- SCOPE 0.9.6 available in the active Python environment.
- The complete SCO ATLAS `Data` directory.
- The local SCOPE environment file `scope_env_atlas.npy` supplied with the data.

Only `data_path` and `env_path` need to be adapted for another local copy.

In [6]:
from pathlib import Path

import csv
import os

import numpy as np
import scope
import scope_sco


In [ ]:
# User configuration
# ------------------
data_path     = Path("/Users/sergivela/Documents/SCOPE/Database_SCO/Data").expanduser().resolve()
env_path      = Path("/Users/sergivela/Documents/SCOPE/Database_SCO/Data/scope_env_atlas.npy").expanduser().resolve()
analysis_path = Path("/Users/sergivela/Documents/SCOPE/Database_SCO/Data").expanduser().resolve()

# Project paths derived from the configured Data directory.
project_path = data_path.parent
root_path    = project_path / "Analysis/1-Analyse_Database"
outputs_path = project_path / "Outputs"

required_scope_version = "0.9.6"
installed_scope_version = getattr(scope, "__version__", None)
if installed_scope_version != required_scope_version:
    raise RuntimeError(
        f"This analysis requires SCOPE {required_scope_version}; "
        f"the active environment provides {installed_scope_version!r}."
    )

required_data_paths = [
    data_path / "1-Sources",
    data_path / "2-Systems",
    data_path / "3-Computations",
]
missing_paths = [path for path in required_data_paths if not path.is_dir()]
if missing_paths:
    raise FileNotFoundError(f"Missing required SCO ATLAS data directories: {missing_paths}")
if not env_path.is_file():
    raise FileNotFoundError(f"SCOPE environment file not found: {env_path}")

outputs_path.mkdir(parents=True, exist_ok=True)
env = scope.read_write.load_binary(str(env_path))

# The serialized environment may retain paths from the machine on which it was created.
env.sources_path = str(data_path / "1-Sources") + os.sep
env.systems_path = str(data_path / "2-Systems") + os.sep
env.computations_path = str(data_path / "3-Computations") + os.sep
env.filepath = str(env_path)


def resolve_system_path(path_from_csv):
    """Resolve a stored system path against the configured local Data directory."""
    path = Path(path_from_csv).expanduser()
    if path.is_file():
        return path.resolve()

    normalized = str(path).replace("\\", "/")
    if "/Data/" in normalized:
        normalized = normalized.split("/Data/", 1)[1]
    elif normalized.startswith("Data/"):
        normalized = normalized[len("Data/"):]
    return data_path / normalized.lstrip("/")


print(f"SCOPE version: {installed_scope_version}")
print(f"Data directory: {data_path}")
print(f"SCOPE environment: {env_path}")
print(f"Analysis outputs: {outputs_path}")


## 1) Preparation of Systems

### 1.1) All systems and Finished ones based on Isolated Branch

In [ ]:
overwrite = False
finished_csv_path     = outputs_path / "finished_systems.csv"
not_finished_csv_path = outputs_path / "not_finished_systems.csv"

if os.path.exists(not_finished_csv_path) and os.path.exists(finished_csv_path) and not overwrite:
    print("LOADING SYSTEMS FROM CSV FILES")
    finished_systems = []
    not_finished = []

    with open(finished_csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sys = scope.read_write.load_binary(str(resolve_system_path(row["path"])))
            sys.set_paths_from_environment(env)
            finished_systems.append(sys)

    with open(not_finished_csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sys = scope.read_write.load_binary(str(resolve_system_path(row["path"])))
            sys.set_paths_from_environment(env)
            not_finished.append(sys)

    systems = finished_systems + not_finished
else:
    from scope.utils.core_tools import get_status
    print("Loading Systems from Environment")
    systems = env.get_all_systems()
    print(len(systems), "systems found in Environment")

    print("FILES NOT FOUND or OVERWRITE. GATHERING SYSTEMS")
    finished_systems = []
    not_finished = []
    finished_rows = []
    not_finished_rows = []
    done = 0
    for sys in systems:
        sys.set_paths_from_environment(env)
        relative_path       = f"Data/2-Systems/{sys.refcode}/{sys.refcode}.npy"
        core = sys.refcode

        ## Makes sure output lines are not stored
        for branch in sys.branches:
            for wf in branch.workflows:
                for job in wf.jobs:
                    for comp in job.computations:
                        if hasattr(comp,"output"): comp.delete_output()  # Delete output to save space 


        ## Makes sure isolated branch
        found, iso_branch = sys.find_branch("isolated")
        if not found:
            not_finished.append(sys)
            reason = f"System {sys.refcode} is missing isolated branch."
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
            continue

        ## Makes sure sources exist
        foundHS, HS_ref_mol = sys.find_source("ref_hs_mol")
        foundLS, LS_ref_mol = sys.find_source("ref_ls_mol")
        if not (foundHS and foundLS):
            not_finished.append(sys)
            reason = f"System {sys.refcode} is missing HS or LS reference molecule."
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
            continue

        ## Total denticity of HS and LS structures should be 6+6
        total_denticity = 0
        for lig in HS_ref_mol.ligands:
            total_denticity += lig.get_denticity()
        for lig in LS_ref_mol.ligands:
            total_denticity += lig.get_denticity()
        if total_denticity != 12:
            not_finished.append(sys)
            reason = f"Denticity != 12"
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
            continue

        ## Makes sure states exist. If not, system is added to not_finished and skipped
        HS_found, HS_min = HS_ref_mol.find_state("b3lyp_opt")
        LS_found, LS_min = LS_ref_mol.find_state("b3lyp_opt")
        if not HS_found or not LS_found:
            not_finished.append(sys)
            reason = f"Missing HS or LS b3lyp_opt states"
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
            continue
        if HS_min.check_fragmentation() or LS_min.check_fragmentation():
            not_finished.append(sys)
            reason = f"Fragmented Molecules in HS or LS b3lyp_opt states"
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
            continue

        ## Checks if states are minima, if not looks for corr_opt states and checks those
        is_HS_min = HS_min.check_minimum()
        is_LS_min = LS_min.check_minimum()
        if not is_HS_min: 
            HS_found, HS_min = HS_ref_mol.find_state("corr_opt")
            if HS_found: is_HS_min = HS_min.check_minimum()
            else:        is_HS_min = False 
        if not is_LS_min: 
            LS_found, LS_min = LS_ref_mol.find_state("corr_opt")
            if LS_found: is_LS_min = LS_min.check_minimum()
            else:        is_LS_min = False
        if is_HS_min and is_LS_min:
            iso_branch.set_status('finished')
            HS_min.get_thermal_data(temp=range(10,501,1))
            LS_min.get_thermal_data(temp=range(10,501,1))
            worked, T12_val = scope_sco.sco_functions.get_T12(iso_branch, HS_min, LS_min, flexible=True, overwrite=True, debug=0)

        iso_status = sys.find_branch("isolated")[1].read_status()
        if iso_status == "finished":
            finished_systems.append(sys)
            finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path})
        else:
            not_finished.append(sys)
            reason = f"isolated branch with status {iso_status}."
            not_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
        sys.save()
        done += 1

    print("Finished Systems:", len(finished_systems))
    print("Not Finished Systems:", len(not_finished))
    print("Saving Files in:", outputs_path)

    with open(finished_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["refcode", "path"])
        writer.writeheader()
        writer.writerows(finished_rows)

    with open(not_finished_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["refcode", "path", "reason"])
        writer.writeheader()
        writer.writerows(not_finished_rows)


In [ ]:
print(len(finished_systems), len(not_finished), "systems in finished and not finished lists.")

### 1.2) Discards Systems with unusual (suspicious) HS or LS structure

Structural parameters are calculated for every finished system. Exclusion details are written immediately, while passing structural records are retained in memory and written to `unique_structural_data.csv` after representative selection in Section 1.3.

In [ ]:
from scope_sco.sco_structure import *

In [ ]:
def get_CH_volume(coord: list):
    from scipy.spatial import ConvexHull, QhullError
    coords = np.asarray(coord, dtype=float)
    if coords.shape[0] < 4: return np.nan
    try:
        ## Scipy returns a "hull" class. Here, we only report the volume
        return np.round(float(ConvexHull(coords).volume),2)
    except QhullError:
        return np.nan

In [ ]:
struc_excluded_path = outputs_path / "filtered_structure_systems.csv"

In [ ]:
pass_structure_check = []
pass_structure_rows = []
excluded = []
excluded_rows = []

for sys in finished_systems:
    relative_path = f"Data/2-Systems/{sys.refcode}/{sys.refcode}.npy"

    ## Finds HS and LS optimized states
    HS_found, HS_ref_mol = sys.find_source('ref_hs_mol')
    LS_found, LS_ref_mol = sys.find_source('ref_ls_mol')
    if not HS_found or not LS_found:
        excluded.append(sys)
        reason = 'Missing HS or LS reference molecule'
        excluded_rows.append({'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished', 'reason': reason})
        continue

    HS_found, HS_min = HS_ref_mol.find_state('b3lyp_opt')
    LS_found, LS_min = LS_ref_mol.find_state('b3lyp_opt')
    if not HS_found or not LS_found:
        excluded.append(sys)
        reason = 'Missing HS or LS b3lyp_opt state'
        excluded_rows.append({'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished', 'reason': reason})
        continue

    ## Refreshes State -> Molecule -> Atom parents using the fixed local SCOPE version
    try:
        HS_min.get_molecules(overwrite=True)
        LS_min.get_molecules(overwrite=True)
    except Exception as e:
        excluded.append(sys)
        reason = f'Error while refreshing b3lyp_opt molecules: {e}'
        excluded_rows.append({'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished', 'reason': reason})
        continue

    ## Computes structural parameters
    try:
        HS_dist, _, HS_sig = geom_sco_from_xyz(HS_min.labels, HS_min.coord)
        LS_dist, _, LS_sig = geom_sco_from_xyz(LS_min.labels, LS_min.coord)
        HS_CH = get_CH_volume(HS_min.coord)
        LS_CH = get_CH_volume(LS_min.coord)
        HS_coordination = len(HS_min.molecules[0].metals[0].get_coord_sphere())
        LS_coordination = len(LS_min.molecules[0].metals[0].get_coord_sphere())
    except Exception as e:
        excluded.append(sys)
        reason = f'Error while computing structural parameters: {e}'
        excluded_rows.append({'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished', 'reason': reason})
        continue

    HS_CShM = 'error'
    LS_CShM = 'error'
    HS_FCS_CH = 'error'
    LS_FCS_CH = 'error'
    reasons = []

    if HS_coordination == 6:
        try:
            HS_CShM = np.round(HS_min.molecules[0].metals[0].get_cshm(overwrite=True), 4)
            HS_FCS_CH = get_CH_volume([at.coord for at in HS_min.molecules[0].metals[0].get_coord_sphere()])
        except Exception as e:
            reasons.append(f'Error while computing HS CShM or FCS volume: {e}')
    else:
        reasons.append(f'HS coordination-sphere atom count is {HS_coordination}, not 6')

    if LS_coordination == 6:
        try:
            LS_CShM = np.round(LS_min.molecules[0].metals[0].get_cshm(overwrite=True), 4)
            LS_FCS_CH = get_CH_volume([at.coord for at in LS_min.molecules[0].metals[0].get_coord_sphere()])
        except Exception as e:
            reasons.append(f'Error while computing LS CShM or FCS volume: {e}')
    else:
        reasons.append(f'LS coordination-sphere atom count is {LS_coordination}, not 6')

    ## Applies the same structural criteria to every system
    if float(HS_dist[0]) < 2.0:
        reasons.append('Unconventionally short HS Fe-N distance (<2.0 Angstrom)')
    if float(LS_dist[0]) > 2.15:
        reasons.append('Unconventionally long LS Fe-N distance (>2.15 Angstrom)')
    if float(HS_dist[0]) - float(LS_dist[0]) < 0:
        reasons.append('Negative LS-to-HS Fe-N distance change')

    if len(reasons) > 0:
        excluded.append(sys)
        excluded_rows.append({
            'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished',
            'HS_dFeN': np.round(HS_dist, 4), 'HS_Sigma': np.round(HS_sig, 4),
            'LS_dFeN': np.round(LS_dist, 4), 'LS_Sigma': np.round(LS_sig, 4),
            'HS_CShM': HS_CShM, 'LS_CShM': LS_CShM,
            'HS_CH_volume': HS_CH, 'LS_CH_volume': LS_CH,
            'HS_FCS_CH_volume': HS_FCS_CH, 'LS_FCS_CH_volume': LS_FCS_CH,
            'HS_coordination_number': HS_coordination, 'LS_coordination_number': LS_coordination,
            'dFeN_HS-LS[Angstrom]': np.round(float(HS_dist[0]) - float(LS_dist[0]), 4),
            'reason': '; '.join(reasons),
        })
        continue

    pass_structure_check.append(sys)
    pass_structure_rows.append({
        'id': sys.refcode_wo_digits, 'path': relative_path, 'status': 'finished',
        'HS_dFeN': np.round(HS_dist, 4), 'HS_Sigma': np.round(HS_sig, 4),
        'LS_dFeN': np.round(LS_dist, 4), 'LS_Sigma': np.round(LS_sig, 4),
        'HS_CShM': HS_CShM, 'LS_CShM': LS_CShM,
        'HS_CH_volume': HS_CH, 'LS_CH_volume': LS_CH,
        'HS_FCS_CH_volume': HS_FCS_CH, 'LS_FCS_CH_volume': LS_FCS_CH,
    })

with open(struc_excluded_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'id', 'path', 'status', 'HS_dFeN', 'HS_Sigma',
        'LS_dFeN', 'LS_Sigma', 'HS_CShM', 'LS_CShM',
        'HS_CH_volume', 'LS_CH_volume', 'HS_FCS_CH_volume', 'LS_FCS_CH_volume',
        'HS_coordination_number', 'LS_coordination_number', 'dFeN_HS-LS[Angstrom]', 'reason',
    ], extrasaction='ignore')
    writer.writeheader()
    writer.writerows(excluded_rows)

print(f'Passed structural check: {len(pass_structure_check)}')
print(f'Excluded by structural check: {len(excluded)}')
print(f'Exclusion details written to: {struc_excluded_path}')

### 1.3) Keeps Unique Isolated Systems That FINISHED and passed the Structure Check

In [ ]:
overwrite = True
unique_csv_path     = outputs_path / "unique_systems.csv"
nonunique_csv_path  = outputs_path / "nonunique_systems.csv"
struc_unique_path = outputs_path / "unique_structural_data.csv"

if os.path.exists(unique_csv_path) and not overwrite:
    print("LOADING UNIQUE SYSTEMS FROM CSV FILE")
    unique_systems = []

    with open(unique_csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sys = scope.read_write.load_binary(str(resolve_system_path(row["path"])))
            sys.set_paths_from_environment(env)
            unique_systems.append(sys)
else:
    unique_systems = []
    unique_rows = []
    unique_structural_rows = []
    nonunique_rows = []
    for idx, sys in enumerate(pass_structure_check):
        relative_path = f"Data/2-Systems/{sys.refcode}/{sys.refcode}.npy"
        found = False
        for lo in unique_systems:
            if sys.find_source("ref_hs_mol")[1] == lo.find_source("ref_hs_mol")[1]:
                reason = f"System {sys.refcode} is the same as system {lo.refcode}."
                nonunique_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "reason": reason})
                found = True
                break
        if not found:
            unique_systems.append(sys)
            unique_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path})
            unique_structural_rows.append(pass_structure_rows[idx])

    with open(unique_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["refcode", "path"])
        writer.writeheader()
        writer.writerows(unique_rows)
    
    with open(nonunique_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["refcode", "path", "reason"])
        writer.writeheader()
        writer.writerows(nonunique_rows)

    with open(struc_unique_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            'id', 'path', 'status', 'HS_dFeN', 'HS_Sigma',
            'LS_dFeN', 'LS_Sigma', 'HS_CShM', 'LS_CShM',
            'HS_CH_volume', 'LS_CH_volume', 'HS_FCS_CH_volume', 'LS_FCS_CH_volume',
        ], extrasaction='ignore')
        #writer = csv.DictWriter(f, fieldnames=[
        #    'id', 'path', 'status', 'HS_FeN av.distance', 'HS_Epsylon',
        #    'LS_FeN av.distance', 'LS_Epsylon', 'HS_CShM', 'LS_CShM',
        #    'HS_CH_volume', 'LS_CH_volume', 'HS_FCS_CH_volume', 'LS_FCS_CH_volume',
        #])
        writer.writeheader()
        writer.writerows(unique_structural_rows)

    print(f'Unique systems after structural check: {len(unique_systems)}')
    print(f'Non-unique systems after structural check: {len(nonunique_rows)}')
    print(f'Unique structural data written to: {struc_unique_path}')

In [ ]:
# overwrite = False
# if os.path.exists(str(root_path / "unique_systems.npy")) and not overwrite:
#     print("LOADING UNIQUE SYSTEMS FROM FILE")
#     unique_systems = scope.read_write.load_binary(str(root_path / "unique_systems.npy"))
# else:
#     unique_systems = []
#     excluded = []
#     for sys in finished_systems:
#         found = False
#         for lo in unique_systems:
#             if sys.find_source("ref_hs_mol")[1] == lo.find_source("ref_hs_mol")[1]:
#                 print(f"{sys.refcode} is same as {lo.refcode}")
#                 print(f"\t{sys.find_source("ref_hs_mol")[1].formula}") 
#                 print(f"\t{lo.find_source("ref_hs_mol")[1].formula}") 
#                 found = True
#         if not found: unique_systems.append(sys)
#     scope.read_write.save_binary(unique_systems,str(root_path / "unique_systems.npy"))

In [ ]:
## This is just to check the unique systems in the overall list of systems. 
unique_systems_overall = []
for sys in systems:
    found = False
    for lo in unique_systems_overall:
        if sys.find_source("ref_hs_mol")[1] == lo.find_source("ref_hs_mol")[1]:
            print(f"{sys.refcode} is same as {lo.refcode}")
            found = True
    if not found: unique_systems_overall.append(sys)
print(len(unique_systems_overall))

### 1.4) Systems with T12 close to Room Temperature

In [ ]:
overwrite = False
csv_path = outputs_path / "rt_systems.csv"

rt_systems = []
rows = []

for sys in unique_systems:
    exists, this_branch = sys.find_branch("isolated")
    assert exists, f"System {sys.refcode} is missing isolated branch"
    assert "T12" in this_branch.results.keys(), f"System {sys.refcode} is missing T12 result"

    res = this_branch.results["T12"].value
    if res is None:
        continue

    t12 = float(res)
    if 273 < t12 < 323:
        rt_systems.append(sys)
        rows.append({
            "id": sys.refcode_wo_digits,
            "path": f"Data/2-Systems/{sys.refcode}/{sys.refcode}.npy",
            "t12": t12,
        })

rows.sort(key=lambda row: row["t12"])
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "path", "t12"])
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
overwrite = False
csv_path = outputs_path / "sco_systems.csv"

sco_systems = []
rows = []

for sys in unique_systems:
    exists, this_branch = sys.find_branch("isolated")
    assert exists, f"System {sys.refcode} is missing isolated branch"
    assert "T12" in this_branch.results.keys(), f"System {sys.refcode} is missing T12 result"

    res = this_branch.results["T12"].value
    if res is None:
        continue

    t12 = float(res)
    if 150 < t12 < 350:
        sco_systems.append(sys)
        rows.append({"id": sys.refcode_wo_digits, "path": f"Data/2-Systems/{sys.refcode}/{sys.refcode}.npy", "t12": t12})

rows.sort(key=lambda row: row["id"])
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "path", "t12"])
    writer.writeheader()
    writer.writerows(rows)